# 🤖 RAG Onboarding Chatbot
### Susu Mbok Darmi
**Final Project — AI Bootcamp**

Stack: LangChain · Groq (LLaMA 3.1) · Qdrant Cloud · SentenceTransformers · Streamlit

---

## Cell 1 — Install Libraries

In [1]:
!pip install langchain langchain-community langchain-text-splitters pymupdf sentence-transformers qdrant-client groq rouge-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 — Konfigurasi Perusahaan
> ⚙️ **GANTI BAGIAN INI** sesuai perusahaan

In [3]:
# ============================================================
# ⚙️  KONFIGURASI — GANTI SESUAI PERUSAHAAN
# ============================================================
COMPANY_NAME    = "Susu Mbok Darmi"
FOLDER_PATH     = "/content/drive/MyDrive/Colab Notebooks/FinalProject_AI_Bootcamp/Data/Input/Susu Mbok Darmi"
COLLECTION_NAME = "susu_mbok_darmi"
# ============================================================

print(f"✅ Konfigurasi: {COMPANY_NAME}")
print(f"📁 Folder     : {FOLDER_PATH}")
print(f"🗄️  Collection : {COLLECTION_NAME}")

✅ Konfigurasi: Susu Mbok Darmi
📁 Folder     : /content/drive/MyDrive/Colab Notebooks/FinalProject_AI_Bootcamp/Data/Input/Susu Mbok Darmi
🗄️  Collection : susu_mbok_darmi


## Cell 4 — Load API Keys

In [4]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"]    = userdata.get("GROQ_API_KEY")
os.environ["QDRANT_URL"]      = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"]  = userdata.get("QDRANT_API_KEY")

print("✅ Semua API Key loaded!")

✅ Semua API Key loaded!


## Cell 5 — Load & Chunk PDF

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith(".pdf")]
all_documents = []

for pdf_file in pdf_files:
    pdf_path = os.path.join(FOLDER_PATH, pdf_file)
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["company"] = COMPANY_NAME
    all_documents.extend(docs)
    print(f"✅ {pdf_file} ({len(docs)} halaman)")

print(f"\n📄 Total halaman: {len(all_documents)}")

# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
all_chunks = text_splitter.split_documents(all_documents)
print(f"🧩 Total chunks: {len(all_chunks)}")

/tmp/ipykernel_1023/328719091.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


✅ SOP_Opening_Closing_Outlet_MbokDarmi.pdf (3 halaman)
✅ FAQ_Karyawan_Baru_Dairysta_MbokDarmi.pdf (5 halaman)
✅ Incident_Report_MbokDarmi.pdf (2 halaman)
✅ Kebijakan_Halal_MbokDarmi.pdf (3 halaman)
✅ Template_Absensi_MbokDarmi.pdf (3 halaman)
✅ Panduan_Customer_Complaint_MbokDarmi.pdf (2 halaman)
✅ Surat_Edaran_MbokDarmi.pdf (3 halaman)
✅ Data_Inventory_MbokDarmi.pdf (2 halaman)
✅ Memo_Internal_MbokDarmi.pdf (3 halaman)
✅ Form_Laporan_Harian_MbokDarmi.pdf (3 halaman)
✅ Materi Training Calon Dairysta.pdf (13 halaman)

📄 Total halaman: 42
🧩 Total chunks: 158


## Cell 6 — Load Embedding Model

In [6]:
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model...")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model loaded!")

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


## Cell 7 — Connect & Upload ke Qdrant Cloud

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

# Connect
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)
print("✅ Terhubung ke Qdrant Cloud!")

# Reset collection
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)
    print("🗑️  Collection lama dihapus")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# Buat index untuk filter
qdrant.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="company",
    field_schema=PayloadSchemaType.KEYWORD
)
print("✅ Collection & index siap!")

# Embed & upload
texts     = [chunk.page_content for chunk in all_chunks]
metadatas = [chunk.metadata for chunk in all_chunks]
print(f"⏳ Embedding {len(texts)} chunks...")
embeddings = embedder.encode(texts, show_progress_bar=True)

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": texts[i], **metadatas[i]}
    )
    for i in range(len(texts))
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"\n✅ {len(points)} chunks tersimpan di Qdrant Cloud!")

✅ Terhubung ke Qdrant Cloud!
🗑️  Collection lama dihapus
✅ Collection & index siap!
⏳ Embedding 158 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


✅ 158 chunks tersimpan di Qdrant Cloud!


## Cell 8 — Fungsi RAG Chat

In [8]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def rag_chat(pertanyaan: str, top_k: int = 5):
    # 1. Embed pertanyaan
    query_vector = embedder.encode(pertanyaan).tolist()

    # 2. Cari chunks relevan di Qdrant
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    ).points

    # 3. Susun context
    context = "\n\n".join([r.payload["text"] for r in results])

    # 4. Generate jawaban
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": f"""Kamu adalah asisten onboarding karyawan baru di {COMPANY_NAME}.
Jawab pertanyaan HANYA berdasarkan konteks dokumen internal perusahaan yang diberikan.
Gunakan bahasa Indonesia yang ramah dan mudah dipahami.
Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak tahu."""
            },
            {
                "role": "user",
                "content": f"KONTEKS:\n{context}\n\nPERTANYAAN:\n{pertanyaan}"
            }
        ]
    )
    return response.choices[0].message.content, results

print(f"✅ Fungsi rag_chat siap untuk {COMPANY_NAME}!")

✅ Fungsi rag_chat siap untuk Susu Mbok Darmi!


## Cell 9 — Test Chatbot

In [9]:
# Ganti pertanyaan sesuai kebutuhan
pertanyaan = "Siapa kita dan apa yang kita lakukan?"

jawaban, sources = rag_chat(pertanyaan)
print(f"❓ Pertanyaan: {pertanyaan}")
print(f"\n💬 Jawaban:")
print(jawaban)
print(f"\n📚 Dari {len(sources)} chunks relevan")

❓ Pertanyaan: Siapa kita dan apa yang kita lakukan?

💬 Jawaban:
Halo Dairysta! Saya senang bertemu denganmu hari ini sebagai anggota tim Susu Mbok Darmi. Kami adalah perusahaan F&B; yang berpusat di Bogor, Indonesia, didirikan oleh Mbok Darmi yang sangat menyayangi cucunya. Kami bergerak di bidang produksi dan penjualan susu yang berkualitas tinggi.

Mengenai apa yang kita lakukan, kami bekerja sama sebagai tim untuk menyajikan produk Susu Mbok Darmi yang terbaik kepada pelanggan kita. Dari produksi, distribusi, hingga penjualan, kami semua berperan untuk menciptakan pengalaman yang menyenangkan bagi cucu-cucu Mbok Darmi, yaitu pelanggan kita.

📚 Dari 5 chunks relevan


In [13]:
# ============================================================
# 💬 CELL TEST — Ketik pertanyaan kamu di sini!
# ============================================================

pertanyaan = input(f"❓ Tanya sesuatu tentang {COMPANY_NAME}: ")

jawaban, sources = rag_chat(pertanyaan)

print(f"\n💬 Jawaban:")
print("─" * 60)
print(jawaban)
print("─" * 60)
print(f"📚 Dari {len(sources)} chunks relevan")

❓ Tanya sesuatu tentang Susu Mbok Darmi: sebutkan variant susu mbok darmi

💬 Jawaban:
────────────────────────────────────────────────────────────
Variasi susu Mbok Darmi adalah:

1. FRESH
2. SPECIAL
3. PREMIUM
────────────────────────────────────────────────────────────
📚 Dari 5 chunks relevan


## Cell 10 — EVALUASI

In [11]:
from rouge_score import rouge_scorer

# ============================================================
# 📊 DATA EVALUASI — Pertanyaan + Jawaban Referensi
# ============================================================
eval_data = [
    {
        "pertanyaan": "Apa visi dan misi perusahaan?",
        "referensi": "Perusahaan memiliki visi dan misi yang menjadi landasan operasional dan pelayanan kepada pelanggan."
    },
    {
        "pertanyaan": "Sebutkan produk atau layanan utama perusahaan.",
        "referensi": "Perusahaan menyediakan produk dan layanan utama sesuai bidang usaha yang tercantum di dokumen internal."
    },
    {
        "pertanyaan": "Bagaimana standar pelayanan kepada pelanggan?",
        "referensi": "Karyawan wajib memberikan pelayanan yang ramah, cepat, dan profesional kepada setiap pelanggan."
    },
    {
        "pertanyaan": "Apa saja peraturan kerja karyawan?",
        "referensi": "Karyawan wajib mematuhi peraturan kerja yang ditetapkan perusahaan termasuk jadwal dan standar perilaku."
    },
    {
        "pertanyaan": "Bagaimana struktur organisasi perusahaan?",
        "referensi": "Perusahaan memiliki struktur organisasi yang jelas dengan pembagian tugas dan tanggung jawab tiap jabatan."
    },
]
# ============================================================

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

print(f"📊 Evaluasi ROUGE — {COMPANY_NAME}")
print("=" * 70)

total_r1, total_r2, total_rL = 0, 0, 0

for i, item in enumerate(eval_data):
    jawaban_chatbot, _ = rag_chat(item["pertanyaan"])
    scores = scorer.score(item["referensi"], jawaban_chatbot)

    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rL = scores['rougeL'].fmeasure

    total_r1 += r1
    total_r2 += r2
    total_rL += rL

    print(f"\n❓ Q{i+1}: {item['pertanyaan']}")
    print(f"   ROUGE-1: {r1:.4f} | ROUGE-2: {r2:.4f} | ROUGE-L: {rL:.4f}")

n = len(eval_data)
print("\n" + "=" * 70)
print(f"📈 RATA-RATA ROUGE SCORE — {COMPANY_NAME}")
print(f"   ROUGE-1 : {total_r1/n:.4f}")
print(f"   ROUGE-2 : {total_r2/n:.4f}")
print(f"   ROUGE-L : {total_rL/n:.4f}")
print("=" * 70)

📊 Evaluasi ROUGE — Susu Mbok Darmi

❓ Q1: Apa visi dan misi perusahaan?
   ROUGE-1: 0.3846 | ROUGE-2: 0.1667 | ROUGE-L: 0.3077

❓ Q2: Sebutkan produk atau layanan utama perusahaan.
   ROUGE-1: 0.1935 | ROUGE-2: 0.0690 | ROUGE-L: 0.1935

❓ Q3: Bagaimana standar pelayanan kepada pelanggan?
   ROUGE-1: 0.1471 | ROUGE-2: 0.0303 | ROUGE-L: 0.1176

❓ Q4: Apa saja peraturan kerja karyawan?
   ROUGE-1: 0.0814 | ROUGE-2: 0.0235 | ROUGE-L: 0.0698

❓ Q5: Bagaimana struktur organisasi perusahaan?
   ROUGE-1: 0.1020 | ROUGE-2: 0.0412 | ROUGE-L: 0.0816

📈 RATA-RATA ROUGE SCORE — Susu Mbok Darmi
   ROUGE-1 : 0.1817
   ROUGE-2 : 0.0661
   ROUGE-L : 0.1541
